# Calculate global mortality with parametric bootstrapping

This may require large memory ~40GB.

In [1]:
import os
import xarray as xr
import warnings
from utils.utils import get_scenario_config
from utils.mortality_utils import att_frac
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
TMREL_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "TMREL")
BETA_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "beta_ozone")
BMR_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "BMR_ozone")

In [3]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

In [4]:
# === Set GBD version ===
GBD_version = "GBD23"

In [5]:
# === Calculate the scalar distributions ===
n_samples = 300

# TMREL from GBD23 (uniform distribution)
tmrel_file = f"{GBD_version}_TMREL_{n_samples}_samples_ozone.nc"
tmrel_path = os.path.join(TMREL_DIR, tmrel_file)
tmrel_da = xr.open_dataarray(tmrel_path)

# Beta from RR per 10ppb (normal distribution)
beta_file = f"{GBD_version}_beta_{n_samples}_samples_ozone.nc"
beta_path = os.path.join(BETA_DIR, beta_file)
beta_da = xr.open_dataarray(beta_path)

# Load BMR for each grid point (normal distribution)
bmr_file = f"{GBD_version}_BMR_Country_Mask_COPD_{n_samples}_samples_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "G6-1.5K"

configs = get_scenario_config(model, scenario)
ensemble_members = configs["ensemble_members"]
years = configs["years"]

O3_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "ozone" / "OSDMA8_BC")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / model / "mortality" / "ozone" / "global" / f"{n_samples}_samples")

for ens_num in ensemble_members:
    print(f"Processing ensemble member {ens_num:02d}")
    # {years.stop - 1} from OSDMA8 calculation
    dates = f"{years.start}-{years.stop - 1}"

    # Load ozone data
    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path).astype("float32")
    o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

    del o3_file, o3_path

    # make o3 dask-backed
    o3 = o3.chunk({'lat': 180, 'lon': 360})

    # range doesn't include the final year which is okay
    # because OSDMA8 excludes final year
    for year in years:
        print(f"Processing year {year}")
        o3_year = o3.sel(year=year)
        # Calculation the attributable fraction
        AF = att_frac(o3_year, tmrel_da, beta_da).chunk({"samples": 10})

        POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})
        # Calculate mortality at each grid point for n samples
        M = AF * BMR * POP
        # Calculate the total global mortality
        global_M = M.sum(dim=("lat", "lon"))

        del o3_year, AF, POP, M

        description = ("Global mortality (COPD) due to ozone "
                       " - scripts by A.F. Wells (2025)")
        global_M.attrs["description"] = description
        global_M.attrs["GBD version"] = GBD_version
        global_M.attrs["model"] = model
        global_M.attrs["scenario"] = scenario
        global_M.attrs["ensemble_number"] = ens_num
        global_M.attrs["year"] = year

        out_file = f"Global_mortality_{GBD_version}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{year}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)
        print(f"Saving to {out_path}")
        global_M.to_netcdf(out_path)

        del global_M

    del o3

print("All processing complete.")

Processing ensemble member 01
Processing year 2035
Saving to /glade/work/awells/workflow/CESM2/mortality/ozone/global/300_samples/Global_mortality_GBD23_300samples_CESM2_G6-1.5K_01_2035.nc
Processing year 2036
Saving to /glade/work/awells/workflow/CESM2/mortality/ozone/global/300_samples/Global_mortality_GBD23_300samples_CESM2_G6-1.5K_01_2036.nc
Processing year 2037
Saving to /glade/work/awells/workflow/CESM2/mortality/ozone/global/300_samples/Global_mortality_GBD23_300samples_CESM2_G6-1.5K_01_2037.nc
Processing year 2038
Saving to /glade/work/awells/workflow/CESM2/mortality/ozone/global/300_samples/Global_mortality_GBD23_300samples_CESM2_G6-1.5K_01_2038.nc
Processing year 2039
Saving to /glade/work/awells/workflow/CESM2/mortality/ozone/global/300_samples/Global_mortality_GBD23_300samples_CESM2_G6-1.5K_01_2039.nc
Processing year 2040
Saving to /glade/work/awells/workflow/CESM2/mortality/ozone/global/300_samples/Global_mortality_GBD23_300samples_CESM2_G6-1.5K_01_2040.nc
Processing year 